In [ ]:
from pathlib import Path

import pandas as pd

data_path = Path("../data/raw/fake_job_postings.csv")

if not data_path.exists():
    data_path = Path("data/raw/fake_job_postings.csv")

df = pd.read_csv(data_path)

print("Dataset loaded successfully")
print("Shape:", df.shape)

In [ ]:
df.head(3)

In [ ]:
df.info()


In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary.sort_values(
    by="missing_percentage",
    ascending=False
)

missing_summary

In [ ]:
class_distribution = pd.DataFrame({
    "count": df["fraudulent"].value_counts().sort_index(),
    "percentage": (
        df["fraudulent"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )
})

class_distribution.index = ["Legitimate (0)", "Fraudulent (1)"]

class_distribution

In [ ]:
print("Unique job IDs:", df["job_id"].nunique())
print("Job IDs are unique:", df["job_id"].is_unique)
print("Target values:", sorted(df["fraudulent"].unique()))

columns_without_id = [
    column for column in df.columns
    if column != "job_id"
]

duplicate_count = df.duplicated(
    subset=columns_without_id
).sum()

print("Duplicate rows excluding job_id:", duplicate_count)

In [ ]:
feature_columns = [
    column for column in df.columns
    if column not in ["job_id", "fraudulent"]
]

duplicate_features = df.duplicated(
    subset=feature_columns
).sum()

label_counts_per_posting = (
    df.groupby(feature_columns, dropna=False)["fraudulent"]
    .nunique()
)

conflicting_groups = (label_counts_per_posting > 1).sum()

print("Repeated inputs:", duplicate_features)
print("Identical inputs with conflicting labels:", conflicting_groups)

In [ ]:
df_clean = (
    df.drop_duplicates(subset=feature_columns)
    .reset_index(drop=True)
    .copy()
)

removed_rows = len(df) - len(df_clean)

print("Original shape:", df.shape)
print("Clean shape:", df_clean.shape)
print("Duplicate rows removed:", removed_rows)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_df, temporary_df = train_test_split(
    df_clean,
    test_size=0.30,
    stratify=df_clean["fraudulent"],
    random_state=42
)

validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    stratify=temporary_df["fraudulent"],
    random_state=42
)

print("Training shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
for name, dataset in [
    ("Training", train_df),
    ("Validation", validation_df),
    ("Test", test_df)
]:
    fraud_percentage = dataset["fraudulent"].mean() * 100

    print(
        f"{name}: {len(dataset)} rows, "
        f"{fraud_percentage:.2f}% fraudulent"
    )

In [ ]:
processed_dir = data_path.parent.parent / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(processed_dir / "train.csv", index=False)
validation_df.to_csv(processed_dir / "validation.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

print("Saved files:")
for file_path in sorted(processed_dir.glob("*.csv")):
    print(file_path.name)

In [ ]:
binary_features = [
    "telecommuting",
    "has_company_logo",
    "has_questions"
]

binary_comparison = (
    train_df
    .groupby("fraudulent")[binary_features]
    .mean()
    .mul(100)
    .round(2)
)

binary_comparison.index = [
    "Legitimate (0)",
    "Fraudulent (1)"
]

binary_comparison

In [ ]:
nullable_features = [
    "location",
    "department",
    "salary_range",
    "company_profile",
    "description",
    "requirements",
    "benefits",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

missing_by_class = pd.DataFrame({
    "Legitimate (0)": (
        train_df.loc[
            train_df["fraudulent"] == 0,
            nullable_features
        ]
        .isna()
        .mean()
        .mul(100)
    ),
    "Fraudulent (1)": (
        train_df.loc[
            train_df["fraudulent"] == 1,
            nullable_features
        ]
        .isna()
        .mean()
        .mul(100)
    )
}).round(2)

missing_by_class["Difference"] = (
    missing_by_class["Fraudulent (1)"]
    - missing_by_class["Legitimate (0)"]
).round(2)

missing_by_class = missing_by_class.reindex(
    missing_by_class["Difference"]
    .abs()
    .sort_values(ascending=False)
    .index
)

missing_by_class

In [ ]:
text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

In [ ]:
train_text = (
    train_df[text_columns]
    .fillna("")
    .agg(" ".join, axis=1)
)

print("Number of documents:", len(train_text))
print("First document length:", len(train_text.iloc[0]))
print()
print(train_text.iloc[0][:500])

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

validation_text = (
    validation_df[text_columns]
    .fillna("")
    .agg(" ".join, axis=1)
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_text = vectorizer.fit_transform(train_text)
X_validation_text = vectorizer.transform(validation_text)

y_train = train_df["fraudulent"]
y_validation = validation_df["fraudulent"]

print("Training feature shape:", X_train_text.shape)
print("Validation feature shape:", X_validation_text.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

validation_text = (
    validation_df[text_columns]
    .fillna("")
    .agg(" ".join, axis=1)
)

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_text = vectorizer.fit_transform(train_text)
X_validation_text = vectorizer.transform(validation_text)

y_train = train_df["fraudulent"]
y_validation = validation_df["fraudulent"]

print("Training feature shape:", X_train_text.shape)
print("Validation feature shape:", X_validation_text.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

In [ ]:
vectorizer.fit_transform(train_text)

In [ ]:
X_train_text = vectorizer.fit_transform(train_text)
X_validation_text = vectorizer.transform(validation_text)

print("Training:", X_train_text.shape)
print("Validation:", X_validation_text.shape)
print("Vocabulary:", len(vectorizer.vocabulary_))

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix

y_train = train_df["fraudulent"]
y_validation = validation_df["fraudulent"]

baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train_text, y_train)

baseline_predictions = baseline_model.predict(
    X_validation_text
)

print("Confusion matrix:")
print(confusion_matrix(y_validation, baseline_predictions))

print("\nClassification report:")
print(
    classification_report(
        y_validation,
        baseline_predictions,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0
    )
)

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_text, y_train)

logistic_predictions = logistic_model.predict(
    X_validation_text
)

print("Confusion matrix:")
print(confusion_matrix(y_validation, logistic_predictions))

print("\nClassification report:")
print(
    classification_report(
        y_validation,
        logistic_predictions,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0
    )
)

In [ ]:
balanced_logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

balanced_logistic_model.fit(X_train_text, y_train)

balanced_predictions = balanced_logistic_model.predict(
    X_validation_text
)

print("Confusion matrix:")
print(confusion_matrix(y_validation, balanced_predictions))

print("\nClassification report:")
print(
    classification_report(
        y_validation,
        balanced_predictions,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0
    )
)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

validation_scores = balanced_logistic_model.predict_proba(
    X_validation_text
)[:, 1]

threshold_results = []

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    predictions = (
        validation_scores >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "flagged": predictions.sum()
    })

threshold_table = pd.DataFrame(
    threshold_results
).round(3)

threshold_table

In [ ]:
selected_threshold = 0.60

project_root = data_path.parent.parent.parent
reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

threshold_table.to_csv(
    reports_dir / "text_model_threshold_comparison.csv",
    index=False
)

print("Selected provisional threshold:", selected_threshold)
print("Threshold report saved")

In [ ]:
from sklearn.pipeline import Pipeline

text_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=5000,
            stop_words="english"
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

In [ ]:
text_pipeline.fit(train_text, y_train)

In [ ]:
pipeline_validation_scores = (
    text_pipeline.predict_proba(validation_text)[:, 1]
)

pipeline_predictions = (
    pipeline_validation_scores >= selected_threshold
).astype(int)

print(
    confusion_matrix(
        y_validation,
        pipeline_predictions
    )
)

print(
    classification_report(
        y_validation,
        pipeline_predictions,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0
    )
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer


def combine_text_columns(frame):
    return (
        frame
        .fillna("")
        .agg(" ".join, axis=1)
    )


def create_missing_indicators(frame):
    return frame.isna().astype("int8")

In [ ]:
text_transformer = Pipeline([
    (
        "combine_text",
        FunctionTransformer(
            combine_text_columns,
            validate=False
        )
    ),
    (
        "tfidf",
        TfidfVectorizer(
            max_features=5000,
            stop_words="english"
        )
    )
])

missing_transformer = FunctionTransformer(
    create_missing_indicators,
    validate=False
)

preprocessor = ColumnTransformer([
    (
        "text",
        text_transformer,
        text_columns
    ),
    (
        "binary",
        "passthrough",
        binary_features
    ),
    (
        "missing",
        missing_transformer,
        nullable_features
    )
])

In [ ]:
jobshield_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

jobshield_pipeline.fit(
    train_df,
    y_train
)

In [ ]:
multi_input_scores = (
    jobshield_pipeline.predict_proba(validation_df)[:, 1]
)

multi_input_predictions = (
    multi_input_scores >= 0.50
).astype(int)

print(
    confusion_matrix(
        y_validation,
        multi_input_predictions
    )
)

print(
    classification_report(
        y_validation,
        multi_input_predictions,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0
    )
)

In [ ]:
multi_threshold_results = []

for threshold in [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]:
    predictions = (
        multi_input_scores >= threshold
    ).astype(int)

    multi_threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            predictions,
            zero_division=0
        ),
        "flagged": predictions.sum()
    })

multi_threshold_table = pd.DataFrame(
    multi_threshold_results
).round(3)

multi_threshold_table

In [ ]:
multi_threshold_table.to_csv(
    reports_dir / "multi_input_threshold_comparison.csv",
    index=False
)

model_comparison = pd.DataFrame([
    {
        "model": "Text-only logistic regression",
        "threshold": 0.60,
        "precision": 0.774,
        "recall": 0.822,
        "f1": 0.797
    },
    {
        "model": "Multi-input logistic regression",
        "threshold": 0.60,
        "precision": 0.586,
        "recall": 0.868,
        "f1": 0.700
    }
])

model_comparison.to_csv(
    reports_dir / "model_comparison.csv",
    index=False
)

model_comparison

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

cross_validator = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cross_validation_results = []

for fold, (fold_train_indices, fold_validation_indices) in enumerate(
    cross_validator.split(train_text, y_train),
    start=1
):
    fold_model = clone(text_pipeline)

    fold_train_text = train_text.iloc[fold_train_indices]
    fold_validation_text = train_text.iloc[fold_validation_indices]

    fold_y_train = y_train.iloc[fold_train_indices]
    fold_y_validation = y_train.iloc[fold_validation_indices]

    fold_model.fit(
        fold_train_text,
        fold_y_train
    )

    fold_scores = fold_model.predict_proba(
        fold_validation_text
    )[:, 1]

    fold_predictions = (
        fold_scores >= 0.60
    ).astype(int)

    cross_validation_results.append({
        "fold": fold,
        "precision": precision_score(
            fold_y_validation,
            fold_predictions,
            zero_division=0
        ),
        "recall": recall_score(
            fold_y_validation,
            fold_predictions,
            zero_division=0
        ),
        "f1": f1_score(
            fold_y_validation,
            fold_predictions,
            zero_division=0
        )
    })

cross_validation_table = pd.DataFrame(
    cross_validation_results
).round(3)

cross_validation_table

In [ ]:
cross_validation_summary = (
    cross_validation_table[
        ["precision", "recall", "f1"]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

cross_validation_summary

In [ ]:
cross_validation_table.to_csv(
    reports_dir / "text_model_cross_validation_folds.csv",
    index=False
)

cross_validation_summary.to_csv(
    reports_dir / "text_model_cross_validation_summary.csv"
)

print("Cross-validation reports saved")

In [ ]:
def create_metric_row(
    model_name,
    threshold,
    actual,
    predicted
):
    return {
        "model": model_name,
        "threshold": threshold,
        "precision": precision_score(
            actual,
            predicted,
            zero_division=0
        ),
        "recall": recall_score(
            actual,
            predicted,
            zero_division=0
        ),
        "f1": f1_score(
            actual,
            predicted,
            zero_division=0
        )
    }


experiment_log = pd.DataFrame([
    create_metric_row(
        "Majority baseline",
        None,
        y_validation,
        baseline_predictions
    ),
    create_metric_row(
        "Text logistic - unweighted",
        0.50,
        y_validation,
        logistic_predictions
    ),
    create_metric_row(
        "Text logistic - balanced",
        0.50,
        y_validation,
        balanced_predictions
    ),
    create_metric_row(
        "Text pipeline - balanced",
        0.60,
        y_validation,
        pipeline_predictions
    ),
    create_metric_row(
        "Multi-input pipeline",
        0.50,
        y_validation,
        multi_input_predictions
    ),
    create_metric_row(
        "Multi-input pipeline",
        0.60,
        y_validation,
        (multi_input_scores >= 0.60).astype(int)
    )
]).round(3)

experiment_log.to_csv(
    reports_dir / "experiment_log.csv",
    index=False
)

experiment_log

In [ ]:
from sklearn.metrics import log_loss

unweighted_scores = logistic_model.predict_proba(
    X_validation_text
)[:, 1]

balanced_scores = balanced_logistic_model.predict_proba(
    X_validation_text
)[:, 1]

unweighted_log_loss = log_loss(
    y_validation,
    unweighted_scores
)

balanced_log_loss = log_loss(
    y_validation,
    balanced_scores
)

print(
    "Unweighted model log loss:",
    round(unweighted_log_loss, 4)
)

print(
    "Balanced model log loss:",
    round(balanced_log_loss, 4)
)

In [ ]:
false_positive_cost = 1
false_negative_cost = 5


def calculate_error_cost(actual, predicted):
    tn, fp, fn, tp = confusion_matrix(
        actual,
        predicted
    ).ravel()

    total_cost = (
        false_positive_cost * fp
        + false_negative_cost * fn
    )

    return {
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "error_cost": total_cost
    }


cost_comparison = pd.DataFrame([
    {
        "model": "Text unweighted - 0.50",
        **calculate_error_cost(
            y_validation,
            logistic_predictions
        )
    },
    {
        "model": "Text balanced - 0.50",
        **calculate_error_cost(
            y_validation,
            balanced_predictions
        )
    },
    {
        "model": "Text balanced - 0.60",
        **calculate_error_cost(
            y_validation,
            pipeline_predictions
        )
    },
    {
        "model": "Multi-input - 0.60",
        **calculate_error_cost(
            y_validation,
            (multi_input_scores >= 0.60).astype(int)
        )
    }
])

cost_comparison.to_csv(
    reports_dir / "cost_comparison.csv",
    index=False
)

cost_comparison

In [ ]:
regularization_results = []
regularization_models = {}

for c_value in [0.25, 0.50, 1.00, 2.00, 4.00]:
    candidate_model = clone(text_pipeline)

    candidate_model.set_params(
        classifier__C=c_value
    )

    candidate_model.fit(
        train_text,
        y_train
    )

    candidate_scores = candidate_model.predict_proba(
        validation_text
    )[:, 1]

    candidate_predictions = (
        candidate_scores >= 0.60
    ).astype(int)

    regularization_models[c_value] = candidate_model

    regularization_results.append({
        "C": c_value,
        "precision": precision_score(
            y_validation,
            candidate_predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_validation,
            candidate_predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_validation,
            candidate_predictions,
            zero_division=0
        ),
        "error_cost": calculate_error_cost(
            y_validation,
            candidate_predictions
        )["error_cost"]
    })

regularization_table = pd.DataFrame(
    regularization_results
).round(3)

regularization_table

In [ ]:
c2_cross_validation_results = []

for fold, (fold_train_indices, fold_validation_indices) in enumerate(
    cross_validator.split(train_text, y_train),
    start=1
):
    fold_model = clone(text_pipeline)

    fold_model.set_params(
        classifier__C=2.0
    )

    fold_model.fit(
        train_text.iloc[fold_train_indices],
        y_train.iloc[fold_train_indices]
    )

    fold_scores = fold_model.predict_proba(
        train_text.iloc[fold_validation_indices]
    )[:, 1]

    fold_predictions = (
        fold_scores >= 0.60
    ).astype(int)

    fold_actual = y_train.iloc[
        fold_validation_indices
    ]

    c2_cross_validation_results.append({
        "fold": fold,
        "precision": precision_score(
            fold_actual,
            fold_predictions,
            zero_division=0
        ),
        "recall": recall_score(
            fold_actual,
            fold_predictions,
            zero_division=0
        ),
        "f1": f1_score(
            fold_actual,
            fold_predictions,
            zero_division=0
        )
    })

c2_cross_validation_table = pd.DataFrame(
    c2_cross_validation_results
).round(3)

c2_cross_validation_table

In [ ]:
c2_cross_validation_summary = (
    c2_cross_validation_table[
        ["precision", "recall", "f1"]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

c2_cross_validation_summary

In [ ]:
regularization_table.to_csv(
    reports_dir / "regularization_comparison.csv",
    index=False
)

c2_cross_validation_table.to_csv(
    reports_dir / "c2_cross_validation_folds.csv",
    index=False
)

c2_cross_validation_summary.to_csv(
    reports_dir / "c2_cross_validation_summary.csv"
)

In [ ]:
bigram_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=5000,
            stop_words="english",
            ngram_range=(1, 2)
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

bigram_pipeline.fit(
    train_text,
    y_train
)

bigram_scores = bigram_pipeline.predict_proba(
    validation_text
)[:, 1]

bigram_predictions = (
    bigram_scores >= 0.60
).astype(int)

print(
    confusion_matrix(
        y_validation,
        bigram_predictions
    )
)

print(
    classification_report(
        y_validation,
        bigram_predictions,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0
    )
)

In [ ]:
def cross_validate_text_pipeline(
    model,
    text,
    targets,
    threshold=0.60,
    number_of_folds=5
):
    splitter = StratifiedKFold(
        n_splits=number_of_folds,
        shuffle=True,
        random_state=42
    )

    results = []

    for fold, (train_indices, validation_indices) in enumerate(
        splitter.split(text, targets),
        start=1
    ):
        fold_model = clone(model)

        fold_model.fit(
            text.iloc[train_indices],
            targets.iloc[train_indices]
        )

        scores = fold_model.predict_proba(
            text.iloc[validation_indices]
        )[:, 1]

        predictions = (
            scores >= threshold
        ).astype(int)

        actual = targets.iloc[validation_indices]

        results.append({
            "fold": fold,
            "precision": precision_score(
                actual,
                predictions,
                zero_division=0
            ),
            "recall": recall_score(
                actual,
                predictions,
                zero_division=0
            ),
            "f1": f1_score(
                actual,
                predictions,
                zero_division=0
            )
        })

    return pd.DataFrame(results)

In [ ]:
bigram_cv_table = cross_validate_text_pipeline(
    bigram_pipeline,
    train_text,
    y_train
).round(3)

bigram_cv_summary = (
    bigram_cv_table[
        ["precision", "recall", "f1"]
    ]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

bigram_cv_summary

In [ ]:
bigram_cv_table.to_csv(
    reports_dir / "bigram_cross_validation_folds.csv",
    index=False
)

bigram_cv_summary.to_csv(
    reports_dir / "bigram_cross_validation_summary.csv"
)

In [ ]:
TEXT_COLUMNS = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits",
]

TARGET_COLUMN = "fraudulent"

MAX_TFIDF_FEATURES = 5000
NGRAM_RANGE = (1, 1)

LOGISTIC_REGRESSION_C = 2.0
CLASS_WEIGHT = "balanced"
MAX_ITERATIONS = 1000

DECISION_THRESHOLD = 0.60
RANDOM_STATE = 42

In [ ]:
no_stopword_pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 1),
            stop_words=None
        )
    ),
    (
        "classifier",
        LogisticRegression(
            C=2.0,
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        )
    )
])

no_stopword_pipeline.fit(
    train_text,
    y_train
)

no_stopword_scores = no_stopword_pipeline.predict_proba(
    validation_text
)[:, 1]

no_stopword_predictions = (
    no_stopword_scores >= 0.60
).astype(int)

print(
    confusion_matrix(
        y_validation,
        no_stopword_predictions
    )
)

print(
    classification_report(
        y_validation,
        no_stopword_predictions,
        target_names=["Legitimate", "Fraudulent"],
        zero_division=0
    )
)